# 🏛️ Model Kolektibilitas WP DSPC — 02 · Modelling

Notebook ini melatih model **prediksi tingkat kolektibilitas WP** (0 = Rendah, 1 = Sedang, 2 = Tinggi) dari hasil pre-processing notebook 01 (`dataset/processed/wp_features.csv`).

**Konteks desain (dari notebook 01):**
- Unit baris: **1 WP** (2.873 baris, kelas seimbang ± 983/939/951).
- Skenario **B — triase portofolio berjalan**: fitur as-of-snapshot (kondisi pembayaran, riwayat tindakan penagihan, kepatuhan SPT, faktur) semuanya sah.
- Split stratified biasa (1 baris = 1 WP), `random_state=42` — **split identik dengan notebook 01**.

**Metrik utama:** `f1_macro` (rata-rata F1 tiga kelas) + **recall per kelas** (kelas Rendah = kasus yang butuh tindakan penagihan paling intensif).

**Model yang dibandingkan:**
| Model | Alasan |
|---|---|
| `DummyClassifier` | baseline minimal (harus dikalahkan) |
| `LogisticRegression` | baseline linear, interpretable |
| `RandomForestClassifier` | pohon ensemble, kuat di tabular |
| `HistGradientBoostingClassifier` | gradient boosting gaya LightGBM (native sklearn) |

> ⚠️ **Catatan jujur:** `LABEL` kemungkinan besar ditetapkan dari aturan atas kolom pembayaran/penagihan — model berpeluang belajar pemetaan itu kembali. Performa tinggi = otomatisasi penilaian yang konsisten, bukan penemuan pola baru; interpretasi (§6) akan memperlihatkan fitur mana yang dominan.

## ⚙️ Setup & Muat Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, balanced_accuracy_score, recall_score,
)
from sklearn.inspection import permutation_importance

RNG = 42
np.random.seed(RNG)
pd.set_option("display.max_columns", 60)

CONFIG = {
    "path_features": "../dataset/processed/wp_features.csv",
    "path_model":    "../models/kolektibilitas_wp.joblib",
    "path_prediksi": "../dataset/processed/test_predictions.csv",
    "label_col": "LABEL",
    "label_names": {0: "Rendah", 1: "Sedang", 2: "Tinggi"},
    "test_size": 0.20,
    "cv_folds": 5,
}

wp = pd.read_csv(CONFIG["path_features"])
print(f"Data   : {wp.shape[0]:,} WP × {wp.shape[1]} kolom")
print("Label  :", wp[CONFIG['label_col']].value_counts().sort_index().to_dict())
wp.head(3)

## 1. 🔀 Persiapan Fitur & Split

Daftar fitur **identik dengan notebook 01** (sinkron manual — bila notebook 01 berubah, sesuaikan daftar ini).

In [ ]:
NUM_FEATURES = [
    # Portofolio tunggakan WP
    "TOTAL_TUNGGAKAN_POKOK", "LOG_TUNGGAKAN_POKOK", "NILAI_TUNGGAKAN_SISA",
    "JML_KETETAPAN", "JML_JENIS_PAJAK", "JML_JENIS_KETETAPAN", "RATA_SELISIH_TAHUN_TERBIT",
    # Kondisi pembayaran & penagihan (as-of-snapshot)
    "TOTAL_NILAI_CAIR", "LOG_NILAI_CAIR", "RASIO_CAIR", "RASIO_SISA",
    "SETOR_SEBELUM_COLL_DATE", "SETOR_SEBELUM_TEGURAN", "SETOR_TEGURAN",
    "SETOR_PAKSA", "SETOR_SITA", "SETOR_CEGAH", "SETOR_SPRINDRA",
    "JML_SURAT_PAKSA",
    "FLAG_PERNAH_DISITA", "FLAG_PERNAH_BLOKIR", "FLAG_RESPON_PENAGIHAN",
    # Durasi & keberadaan tindakan
    "UMUR_TUNGGAKAN_HARI", "UMUR_INKRAH_HARI", "SISA_DALUWARSA_HARI", "FLAG_DALUWARSA_DEKAT",
    "HARI_SEJAK_TEGURAN", "HARI_SEJAK_SP", "HARI_SEJAK_BAPS",
    "FLAG_TEGURAN_ADA", "FLAG_SP_ADA", "FLAG_BAPS_ADA",
    # Kepatuhan & aktivitas ekonomi
    "RASIO_LAPOR_SPT_3THN", "FLAG_LAPOR_SPT_TERAKHIR",
    "PEREDARAN_BRUTO", "LOG_PEREDARAN_BRUTO", "FLAG_PEREDARAN_NOL",
    "RASIO_TUNGGAKAN_PEREDARAN",
    "JML_CUSTOMER", "DPP_CUSTOMER", "LOG_DPP_CUSTOMER", "FAKTUR_CUSTOMER",
    "JML_SUPPLIER", "DPP_SUPPLIER", "LOG_DPP_SUPPLIER", "FAKTUR_SUPPLIER",
    "TOTAL_MITRA",
]
CAT_FEATURES = ["STS_WP", "JENIS_WP", "JENIS_KPP_BKM", "KD_KANWIL", "SEKTOR_KLU"]
FEATURES = NUM_FEATURES + CAT_FEATURES

LAB = CONFIG["label_col"]
X = wp[FEATURES].copy()
y = wp[LAB].copy()

# Split identik dengan notebook 01 (seed & test_size sama, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG["test_size"], stratify=y, random_state=RNG,
)
print(f"Train: {X_train.shape} ({y_train.mean()*100:.1f}% )  |  Test: {X_test.shape}")
print("Proporsi kelas train:", (y_train.value_counts(normalize=True).sort_index() * 100).round(1).to_dict())


def buat_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), NUM_FEATURES),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), CAT_FEATURES),
        ],
        remainder="drop",
    )

## 2. ⚖️ Perbandingan Model (Cross-Validation)

5-fold `StratifiedKFold` pada **train** saja (preprocessor di-fit ulang per fold di dalam pipeline → tidak ada leakage antar fold).

In [ ]:
def buat_model(nama: str):
    if nama == "Dummy":
        return DummyClassifier(strategy="most_frequent")
    if nama == "LogReg":
        return LogisticRegression(max_iter=2000, random_state=RNG)
    if nama == "RandomForest":
        return RandomForestClassifier(n_estimators=400, n_jobs=-1, random_state=RNG)
    if nama == "HistGB":
        return HistGradientBoostingClassifier(random_state=RNG)
    raise ValueError(nama)


models = ["Dummy", "LogReg", "RandomForest", "HistGB"]
cv = StratifiedKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=RNG)

rows = []
for nama in models:
    pipe = Pipeline([("prep", buat_preprocessor()), ("clf", buat_model(nama))])
    sc = cross_validate(pipe, X_train, y_train, cv=cv,
                        scoring=["f1_macro", "balanced_accuracy", "accuracy"], n_jobs=-1)
    rows.append({
        "model": nama,
        "f1_macro": sc["test_f1_macro"].mean(),
        "f1_macro_std": sc["test_f1_macro"].std(),
        "balanced_acc": sc["test_balanced_accuracy"].mean(),
        "accuracy": sc["test_accuracy"].mean(),
        "fit_time_s": sc["fit_time"].mean(),
    })
hasil_cv = pd.DataFrame(rows).set_index("model").sort_values("f1_macro", ascending=False)
display(hasil_cv.round(4))

fig, ax = plt.subplots(figsize=(6, 3))
hasil_cv["f1_macro"].sort_values().plot(kind="barh", ax=ax, color="steelblue", xerr=hasil_cv["f1_macro_std"])
ax.set_xlabel("F1-macro (CV 5-fold, train)")
ax.set_title("Perbandingan model")
plt.tight_layout(); plt.show()

## 3. 🔧 Tuning Ringan Model Terbaik

`RandomizedSearchCV` (15 kombinasi × 3-fold) pada model pemenang CV — dioptimalkan untuk `f1_macro`.

In [ ]:
param_grids = {
    "LogReg": {
        "clf__C": [0.05, 0.1, 0.5, 1, 5, 10],
        "clf__class_weight": [None, "balanced"],
    },
    "RandomForest": {
        "clf__n_estimators": [300, 500, 800],
        "clf__max_depth": [None, 12, 20],
        "clf__min_samples_leaf": [1, 2, 5],
        "clf__max_features": ["sqrt", 0.5],
    },
    "HistGB": {
        "clf__learning_rate": [0.05, 0.1, 0.2],
        "clf__max_iter": [200, 400, 600],
        "clf__max_leaf_nodes": [15, 31, 63],
        "clf__min_samples_leaf": [10, 20, 40],
        "clf__l2_regularization": [0.0, 1.0],
    },
}

model_terbaik = hasil_cv.index[0]
print(f"Model CV terbaik: {model_terbaik} -> tuning")

search = RandomizedSearchCV(
    Pipeline([("prep", buat_preprocessor()), ("clf", buat_model(model_terbaik))]),
    param_distributions=param_grids[model_terbaik],
    n_iter=15, cv=StratifiedKFold(3, shuffle=True, random_state=RNG),
    scoring="f1_macro", n_jobs=-1, random_state=RNG,
)
search.fit(X_train, y_train)
print(f"Skor CV terbaik (f1_macro): {search.best_score_:.4f}")
print("Parameter terbaik          :", search.best_params_)

final_model = search.best_estimator_

## 4. 🎯 Evaluasi Final (Test Set)

Test set **belum pernah disentuh** pada tahap mana pun (split identik notebook 01).

In [ ]:
y_pred = final_model.predict(X_test)

print("Classification report (test):")
print(classification_report(y_test, y_pred, target_names=[CONFIG["label_names"][k] for k in sorted(y_test.unique())]))

f1m = f1_score(y_test, y_pred, average="macro")
bal = balanced_accuracy_score(y_test, y_pred)
rec = recall_score(y_test, y_pred, average=None, labels=[0, 1, 2])
print(f"F1-macro          : {f1m:.4f}")
print(f"Balanced accuracy : {bal:.4f}")
print(f"Recall Rendah/Sedang/Tinggi : {rec.round(3)}")

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, ax=ax, normalize="true", cmap="Blues",
    display_labels=[CONFIG["label_names"][k] for k in [0, 1, 2]],
)
ax.set_title("Confusion matrix (normalisasi per kelas)")
plt.tight_layout(); plt.show()

## 5. 🔍 Analisis Segmen

Dua segmen yang paling relevan untuk deploy:
1. **Status WP** (`AKTIF` vs `NON-AKTIF` = NE/DE/UNKNOWN) — pastikan model tidak hanya belajar dari kasus AKTIF.
2. **Outstanding vs lunas** (`NILAI_TUNGGAKAN_SISA > 0`) — saat triase, skor dipakai pada kasus outstanding.

In [ ]:
def metrik_segmen(mask, nama):
    yt, yp = y_test[mask], y_pred[mask]
    if len(yt) < 5 or yt.nunique() < 2:
        return None
    r = recall_score(yt, yp, average=None, labels=[0, 1, 2], zero_division=0)
    return {
        "segmen": nama, "n": int(mask.sum()),
        "f1_macro": f1_score(yt, yp, average="macro", zero_division=0),
        "recall_Rendah": r[0], "recall_Sedang": r[1], "recall_Tinggi": r[2],
    }

sts_test = wp.loc[X_test.index, "STS_WP"]
sisa_test = wp.loc[X_test.index, "NILAI_TUNGGAKAN_SISA"]

segmen = [
    metrik_segmen(np.ones(len(y_test), dtype=bool), "SEMUA (test)"),
    metrik_segmen((sts_test == "AKTIF").values, "WP AKTIF"),
    metrik_segmen((sts_test != "AKTIF").values, "WP NON-AKTIF (NE/DE)"),
    metrik_segmen((sisa_test > 0).values, "Outstanding (sisa>0)"),
    metrik_segmen((sisa_test <= 0).values, "Lunas (sisa=0)"),
]
display(pd.DataFrame([s for s in segmen if s]).round(3))

## 6. 🧠 Interpretasi — Permutation Importance

Permutation importance diukur pada **test set** (drop F1-macro saat satu fitur diacak) — lebih jujur daripada impurity importance yang bias ke fitur kardinalitas tinggi.

In [ ]:
%%time
perm = permutation_importance(
    final_model, X_test, y_test, scoring="f1_macro",
    n_repeats=10, random_state=RNG, n_jobs=-1,
)
imp = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 7))
imp.head(20)[::-1].plot(kind="barh", ax=ax, color="seagreen")
ax.set_title("Top-20 fitur — permutation importance (test, F1-macro)")
ax.set_xlabel("Penurunan F1-macro saat fitur diacak")
plt.tight_layout(); plt.show()

display(imp.head(15).round(4).to_frame("importance"))

## 7. 📋 Demo Triase — Prioritisasi WP Outstanding

Ilustrasi pemakaian: untuk WP **outstanding** di test set, hitung probabilitas tiap kelas lalu susun dua daftar kerja:
- **Aksi intensif** — `P(Rendah)` tertinggi & sisa tunggakan besar (butuh eskalasi: sita/blokir/SPRINDRA),
- **Quick win** — `P(Tinggi)` tertinggi & sisa besar (cukup teguran/pengingat, peluang cair tinggi).

In [ ]:
proba = final_model.predict_proba(X_test)
kelas = final_model.named_steps["clf"].classes_
df_proba = pd.DataFrame(proba, columns=[f"P_{CONFIG['label_names'][k]}" for k in kelas], index=X_test.index)

tri = wp.loc[X_test.index, ["NPWP16", "NAMA_WP", "STS_WP", "NILAI_TUNGGAKAN_SISA", "LABEL"]].join(df_proba)
tri_out = tri[tri["NILAI_TUNGGAKAN_SISA"] > 0].copy()
print(f"WP outstanding di test: {len(tri_out):,}")

kol = ["NPWP16", "NILAI_TUNGGAKAN_SISA", "P_Rendah", "P_Sedang", "P_Tinggi", "LABEL"]

print("\n🔺 Aksi intensif — top-8 P(Rendah) dengan sisa terbesar:")
aksi = tri_out.sort_values(["P_Rendah", "NILAI_TUNGGAKAN_SISA"], ascending=False).head(8)
display(aksi[kol].round(3))

print("\n✅ Quick win — top-8 P(Tinggi) dengan sisa terbesar:")
win = tri_out.sort_values(["P_Tinggi", "NILAI_TUNGGAKAN_SISA"], ascending=False).head(8)
display(win[kol].round(3))

## 8. 💾 Simpan Model & Prediksi

In [ ]:
import os

os.makedirs(os.path.dirname(CONFIG["path_model"]), exist_ok=True)
joblib.dump(final_model, CONFIG["path_model"])

pred_full = tri.copy()
pred_full["PREDIKSI"] = final_model.predict(X_test)
pred_full.to_csv(CONFIG["path_prediksi"], index=False)

print(f"Model tersimpan    : {CONFIG['path_model']}")
print(f"Prediksi test      : {CONFIG['path_prediksi']} ({len(pred_full):,} baris)")

# Contoh inference ulang:
# loaded = joblib.load(CONFIG["path_model"])
# loaded.predict(X_test.head(3)), loaded.predict_proba(X_test.head(3))

## 📌 Ringkasan & Langkah Berikutnya

**Hasil notebook ini:**
- Perbandingan 4 model via CV stratified → tuning ringan pemenang → evaluasi final pada test set (split identik notebook 01).
- Metrik: F1-macro utama + recall per kelas; analisis segmen `STS_WP` & outstanding-vs-lunas.
- Interpretasi via permutation importance pada test set.
- Demo triase: dua daftar kerja (aksi intensif vs quick win) untuk WP outstanding.
- Artefak: `models/kolektibilitas_wp.joblib` + `dataset/processed/test_predictions.csv`.

**Langkah berikutnya:**
- [ ] **SHAP** untuk penjelasan per-WP (akuntabilitas keputusan penagihan) — `pip install shap`
- [ ] **Threshold/biaya per kelas** — salah prediksi Rendah-vs-Tinggi lebih mahal daripada Sedang; turunkan/matkai threshold sesuai biaya
- [ ] **Kalibrasi probabilitas** (`CalibratedClassifierCV`) bila probabilitas dipakai untuk ranking anggaran
- [ ] **Validasi aturan LABEL bersama tim domain** — konfirmasi dominasi fitur pembayaran (§6) memang cerminan aturan penilaian, bukan artefak
- [ ] **Monitoring drift** — skor ulang berkala; distribusi fitur & performa bisa bergeser saat komposisi portofolio berubah
- [ ] **Retraining terjadwal** dengan tarikan data baru (populasi dikontrol tanggal tarikan seperti sekarang)